# Exploratory Data Analysis

## Setup

Imports and read the csv file

In [143]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8")

# Load the dataset
building_consumption = pd.read_csv("../data/raw/2016_Building_Energy_Benchmarking.csv")

## Scope and target

**Goal:** predict the energy consumption of **non-residential** buildings.

- **Target:** `SiteEnergyUse(kBtu)` (total energy consumed on site).
- **Scope:** the file mixes multifamily housing (~1,700 rows) with non-residential buildings (~1,670 rows), so we have to filter.
- **Leakage:** many columns are computed from the energy bills (EUI, electricity, gas, steam, GHG). They cannot be used as features: for a building with no consumption data yet, we would not have them.

In [144]:
# Display the first few rows of the dataset
building_consumption.head()

,OSEBuildingID,DataYear,BuildingType,PrimaryPropertyType,PropertyName,Address,City,State,ZipCode,TaxParcelIdentificationNumber,...,Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),DefaultData,Comments,ComplianceStatus,Outlier,TotalGHGEmissions,GHGEmissionsIntensity
0,1,2016,NonResidential,Hotel,Mayflower park hotel,405 Olive way,Seattle,WA,98101.0,0659000030,...,1.156514e+06,3946027.0,12764.52930,1276453.0,False,NaN,Compliant,NaN,249.98,2.83
1,2,2016,NonResidential,Hotel,Paramount Hotel,724 Pine street,Seattle,WA,98101.0,0659000220,...,9.504252e+05,3242851.0,51450.81641,5145082.0,False,NaN,Compliant,NaN,295.86,2.86
2,3,2016,NonResidential,Hotel,5673-The Westin Seattle,1900 5th Avenue,Seattle,WA,98101.0,0659000475,...,1.451544e+07,49526664.0,14938.00000,1493800.0,False,NaN,Compliant,NaN,2089.28,2.19
3,5,2016,NonResidential,Hotel,HOTEL MAX,620 STEWART ST,Seattle,WA,98101.0,0659000640,...,8.115253e+05,2768924.0,18112.13086,1811213.0,False,NaN,Compliant,NaN,286.43,4.67
4,8,2016,NonResidential,Hotel,WARWICK SEATTLE HOTEL (ID8),401 LENORA ST,Seattle,WA,98121.0,0659000970,...,1.573449e+06,5368607.0,88039.98438,8803998.0,False,NaN,Compliant,NaN,505.01,2.88


In [145]:
building_consumption.info()

<class 'pandas.DataFrame'>
RangeIndex: 3376 entries, 0 to 3375
Data columns (total 46 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   OSEBuildingID                    3376 non-null   int64  
 1   DataYear                         3376 non-null   int64  
 2   BuildingType                     3376 non-null   str    
 3   PrimaryPropertyType              3376 non-null   str    
 4   PropertyName                     3376 non-null   str    
 5   Address                          3376 non-null   str    
 6   City                             3376 non-null   str    
 7   State                            3376 non-null   str    
 8   ZipCode                          3360 non-null   float64
 9   TaxParcelIdentificationNumber    3376 non-null   str    
 10  CouncilDistrictCode              3376 non-null   int64  
 11  Neighborhood                     3376 non-null   str    
 12  Latitude                       

In [146]:
building_consumption.describe()

,OSEBuildingID,DataYear,ZipCode,CouncilDistrictCode,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,...,SiteEnergyUse(kBtu),SiteEnergyUseWN(kBtu),SteamUse(kBtu),Electricity(kWh),Electricity(kBtu),NaturalGas(therms),NaturalGas(kBtu),Comments,TotalGHGEmissions,GHGEmissionsIntensity
count,3376.000000,3376.0,3360.000000,3376.000000,3376.000000,3376.000000,3376.000000,3368.000000,3376.000000,3.376000e+03,...,3.371000e+03,3.370000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3.367000e+03,3.367000e+03,0.0,3367.000000,3367.000000
mean,21208.991114,2016.0,98116.949107,4.439277,47.624033,-122.334795,1968.573164,1.106888,4.709123,9.483354e+04,...,5.403667e+06,5.276726e+06,2.745959e+05,1.086639e+06,3.707612e+06,1.368505e+04,1.368505e+06,NaN,119.723971,1.175916
std,12223.757015,0.0,18.615205,2.120625,0.047758,0.027203,33.088156,2.108402,5.494465,2.188376e+05,...,2.161063e+07,1.593879e+07,3.912173e+06,4.352478e+06,1.485066e+07,6.709781e+04,6.709781e+06,NaN,538.832227,1.821452
min,1.000000,2016.0,98006.000000,1.000000,47.499170,-122.414250,1900.000000,0.000000,0.000000,1.128500e+04,...,0.000000e+00,0.000000e+00,0.000000e+00,-3.382680e+04,-1.154170e+05,0.000000e+00,0.000000e+00,NaN,-0.800000,-0.020000
25%,19990.750000,2016.0,98105.000000,3.000000,47.599860,-122.350662,1948.000000,1.000000,2.000000,2.848700e+04,...,9.251286e+05,9.701822e+05,0.000000e+00,1.874229e+05,6.394870e+05,0.000000e+00,0.000000e+00,NaN,9.495000,0.210000
50%,23112.000000,2016.0,98115.000000,4.000000,47.618675,-122.332495,1975.000000,1.000000,4.000000,4.417500e+04,...,1.803753e+06,1.904452e+06,0.000000e+00,3.451299e+05,1.177583e+06,3.237538e+03,3.237540e+05,NaN,33.920000,0.610000
75%,25994.250000,2016.0,98122.000000,7.000000,47.657115,-122.319407,1997.000000,1.000000,5.000000,9.099200e+04,...,4.222455e+06,4.381429e+06,0.000000e+00,8.293178e+05,2.829632e+06,1.189033e+04,1.189034e+06,NaN,93.940000,1.370000
max,50226.000000,2016.0,98272.000000,7.000000,47.733870,-122.220966,2015.000000,111.000000,99.000000,9.320156e+06,...,8.739237e+08,4.716139e+08,1.349435e+08,1.925775e+08,6.570744e+08,2.979090e+06,2.979090e+08,NaN,16870.980000,34.090000


### A look at the missing values

In [147]:
# Share of missing values per column (only columns with at least one NA)
missing = building_consumption.isna().mean().sort_values(ascending=False)
missing[missing > 0].mul(100).round(1).rename("% missing")

Comments                           100.0
Outlier                             99.1
YearsENERGYSTARCertified            96.5
ThirdLargestPropertyUseType         82.3
ThirdLargestPropertyUseTypeGFA      82.3
SecondLargestPropertyUseType        50.3
SecondLargestPropertyUseTypeGFA     50.3
ENERGYSTARScore                     25.0
LargestPropertyUseTypeGFA            0.6
LargestPropertyUseType               0.6
ZipCode                              0.5
ListOfAllPropertyUseTypes            0.3
Electricity(kWh)                     0.3
SourceEUIWN(kBtu/sf)                 0.3
GHGEmissionsIntensity                0.3
TotalGHGEmissions                    0.3
NaturalGas(therms)                   0.3
SteamUse(kBtu)                       0.3
NaturalGas(kBtu)                     0.3
SourceEUI(kBtu/sf)                   0.3
Electricity(kBtu)                    0.3
NumberofBuildings                    0.2
SiteEUI(kBtu/sf)                     0.2
SiteEnergyUseWN(kBtu)                0.2
SiteEUIWN(kBtu/s

## Column triage

We sort the 46 columns into four groups:

1. **Target**: what we predict.
2. **Key features**: physical or structural properties known without any energy bill. They drive consumption.
3. **Leakage**: derived from the target, so we drop them.
4. **Useless or redundant**: constant, empty, identifiers, free text or duplicated information, with no predictive value.

Some columns are kept only as feature-engineering candidates (`FE_CANDIDATES`, e.g. the use-type columns).

### Is `PropertyGFATotal` redundant?

In [148]:
gfa_total_match_rate = building_consumption.eval(
    "PropertyGFATotal == `PropertyGFABuilding(s)` + PropertyGFAParking"
).mean()
print(f"Match rate for PropertyGFATotal: {gfa_total_match_rate:.2%}")

Match rate for PropertyGFATotal: 100.00%


**Decision: drop `PropertyGFATotal`.** It is exactly `PropertyGFABuilding(s)` + `PropertyGFAParking` on every row, so it adds no information. We keep the two parts rather than the total, because a square foot of parking uses far less energy than a square foot of building.

### Are the areas by use unreliable and redundant?

`LargestPropertyUseTypeGFA` and `SecondLargestPropertyUseTypeGFA` give the area of each use. Two claims to check, on the full file like the rest of the column triage: the areas can exceed the property area (unreliable), and they carry the same information as the building area (redundant).

In [149]:
total_area = building_consumption["PropertyGFATotal"]
building_area = building_consumption["PropertyGFABuilding(s)"]

use_area_checks = {}
for col in ["LargestPropertyUseTypeGFA", "SecondLargestPropertyUseTypeGFA"]:
    use_area = building_consumption[col]
    use_area_checks[col] = {
        "rows with a value": use_area.notna().sum(),
        "% above PropertyGFATotal": (use_area > total_area)[use_area.notna()].mean()
        * 100,
        "Pearson vs building area": use_area.corr(building_area),
        "Spearman vs building area": use_area.corr(building_area, method="spearman"),
    }

pd.DataFrame(use_area_checks).T.round(2)

,rows with a value,% above PropertyGFATotal,Pearson vs building area,Spearman vs building area
LargestPropertyUseTypeGFA,3356.0,10.1,0.98,0.93
SecondLargestPropertyUseTypeGFA,1679.0,0.3,0.79,0.64


**Decision: drop both area columns, keep the use-type categories.**

- **`LargestPropertyUseTypeGFA`**: both claims hold. The area of one use exceeds the whole property area in 10.1% of the rows (339 of 3,356), which is impossible. It is also almost the building area itself (Pearson 0.98, Spearman 0.93), so it adds little. Pearson is higher because the largest buildings, where both areas are nearly equal, weigh the most; Spearman, computed on ranks, shows a bit more disagreement among smaller buildings.
- **`SecondLargestPropertyUseTypeGFA`**: it is not really redundant. Only 0.3% of the rows (5) exceed the property area, and the correlation with the building area is weaker (Pearson 0.79, Spearman 0.64): it does carry some information of its own, the share of the building used for something else. Its missing values are structural (a single-use building has no second use, so its area is 0), not a reason to drop it. We drop it because it is declarative like the first one, and the useful information, the type of the secondary use, is kept in `SecondLargestPropertyUseType`.

### Weather-normalized or raw target?


In [150]:
# Compare the two versions
raw_wn = building_consumption[["SiteEnergyUse(kBtu)", "SiteEnergyUseWN(kBtu)"]].dropna()
raw, wn = raw_wn.iloc[:, 0], raw_wn.iloc[:, 1]

# Pearson is sensitive to extreme values: check it without the largest row
largest = raw.idxmax()
no_max = raw_wn.drop(largest)
print(
    f"Largest row: {building_consumption.loc[largest, 'PropertyName']} ({raw[largest]:,.0f} kBtu)"
)
print(
    f"Pearson : {raw.corr(wn):.3f}  (without the largest row: {no_max.iloc[:, 0].corr(no_max.iloc[:, 1]):.3f})"
)

# Spearman works on ranks, so a single huge value cannot dominate it
print(f"Spearman: {raw.corr(wn, method='spearman'):.3f}")

Largest row: University of Washington - Seattle Campus (873,923,712 kBtu)
Pearson : 0.715  (without the largest row: 0.997)
Spearman: 0.986


The two versions should be almost the same quantity, yet Pearson is only 0.715. The culprit is a single row, the UW campus: without it, Pearson jumps to 0.997. Spearman, computed on ranks, stays high on all rows (0.986), so the two versions agree for small buildings too, not only for the largest ones.

**Decision: keep the raw `SiteEnergyUse(kBtu)` as the target.** Normalizing for weather would change little. The raw value is what the building actually consumed, while the weather-normalized value is itself the output of a model run by the city. `SiteEnergyUseWN(kBtu)` stays out of the features (leakage).

### Summary

In [151]:
TARGET = "SiteEnergyUse(kBtu)"

# Super important: physical / structural features, known without any energy bill
KEY_FEATURES = {
    "PropertyGFABuilding(s)": "Floor area of the heated/cooled buildings (excludes parking)",
    "PropertyGFAParking": "Parking area: low energy use per sq ft, so it dilutes the total",
    "PrimaryPropertyType": "Coarser building category, fewer levels than LargestPropertyUseType",
    "NumberofFloors": "Building height, correlated with size and HVAC needs",
    "NumberofBuildings": "Campus vs single building",
    "YearBuilt": "candidate for a construction-period feature",
    "Neighborhood": "Cleaner than ZipCode, 13 levels after cleaning, OneHot",
    "Latitude": "Location (microclimate, downtown density)",
    "Longitude": "Location (microclimate, downtown density)",
}

# Feature-engineering candidates: may help, but need a decision
FE_CANDIDATES = {
    "LargestPropertyUseType": "Main activity (office, hospital, warehouse...): EUI varies hugely by use",
    "SecondLargestPropertyUseType": "Secondary use (NaN = single-use building)",
    "ListOfAllPropertyUseTypes": (
        "All declared uses, including minor ones hidden by PrimaryPropertyType (e.g. a data center "
        "inside an office): source for energy-intensive use flags"
    ),
}

# Leakage: computed from the target (or from its components), never use as features
LEAKAGE = {
    "SiteEnergyUseWN(kBtu)": "Weather-normalized version of the target",
    "SiteEUI(kBtu/sf)": "Target / floor area",
    "SiteEUIWN(kBtu/sf)": "Weather-normalized EUI",
    "SourceEUI(kBtu/sf)": "Source energy / floor area",
    "SourceEUIWN(kBtu/sf)": "Weather-normalized source EUI",
    "Electricity(kWh)": "Component of the target",
    "Electricity(kBtu)": "Component of the target (same as kWh, other unit)",
    "NaturalGas(therms)": "Component of the target",
    "NaturalGas(kBtu)": "Component of the target (same as therms, other unit)",
    "SteamUse(kBtu)": "Component of the target",
    "TotalGHGEmissions": "Computed from energy consumption",
    "GHGEmissionsIntensity": "Computed from energy consumption",
    "ENERGYSTARScore": (
        "1-100 percentile rank of the building's measured energy use (EUI) against similar "
        "buildings: it can only be computed once the consumption is known. Also 25% missing"
    ),
}

# Useless: no predictive information
USELESS_OR_REDUNDANT = {
    "BuildingType": "Only used to filter non-residential properties; redundant with PrimaryPropertyType once filtered",
    "Comments": "100% empty",
    "YearsENERGYSTARCertified": "96% empty",
    "ThirdLargestPropertyUseType": "82% empty",
    "ThirdLargestPropertyUseTypeGFA": "82% empty",
    "Outlier": "99% empty: only used to flag rows to drop",
    "DataYear": "Constant (2016)",
    "City": "Constant (Seattle)",
    "State": "Constant (WA)",
    "OSEBuildingID": "Row identifier",
    "PropertyName": "Free text identifier",
    "Address": "Free text identifier, near unique",
    "TaxParcelIdentificationNumber": "Identifier",
    "ZipCode": "Redundant with Neighborhood / Lat-Long",
    "CouncilDistrictCode": "Redundant with Neighborhood / Lat-Long",
    "DefaultData": "Redundant with ComplianceStatus (True <=> 'Error - Correct Default Data'), not used",
    "ComplianceStatus": "Quality flag, only used to filter rows (default data = unreliable features)",
    "PropertyGFATotal": "Redundant: always = PropertyGFABuilding(s) + PropertyGFAParking",
    "LargestPropertyUseTypeGFA": "Unreliable (can exceed the property area) and redundant with the building / parking areas",
    "SecondLargestPropertyUseTypeGFA": "Unreliable (can exceed the property area) and redundant with the building / parking areas",
}

# Safety net: every column must be classified exactly once
classified = [TARGET, *KEY_FEATURES, *FE_CANDIDATES, *LEAKAGE, *USELESS_OR_REDUNDANT]
print("Unclassified:", set(building_consumption.columns) - set(classified))
print("Duplicated:", {c for c in classified if classified.count(c) > 1})

# Overview table
groups = {
    "key": KEY_FEATURES,
    "fe_candidate": FE_CANDIDATES,
    "leakage": LEAKAGE,
    "useless_or_redundant": USELESS_OR_REDUNDANT,
}
triage = pd.DataFrame(
    [
        (group, col, reason)
        for group, cols in groups.items()
        for col, reason in cols.items()
    ],
    columns=["group", "column", "reason"],
)
triage

Unclassified: set()
Duplicated: set()


,group,column,reason
0,key,PropertyGFABuilding(s),Floor area of the heated/cooled buildings (exc...
1,key,PropertyGFAParking,"Parking area: low energy use per sq ft, so it ..."
2,key,PrimaryPropertyType,"Coarser building category, fewer levels than L..."
3,key,NumberofFloors,"Building height, correlated with size and HVAC..."
4,key,NumberofBuildings,Campus vs single building
5,key,YearBuilt,candidate for a construction-period feature
6,key,Neighborhood,"Cleaner than ZipCode, 13 levels after cleaning..."
7,key,Latitude,"Location (microclimate, downtown density)"
8,key,Longitude,"Location (microclimate, downtown density)"
9,fe_candidate,LargestPropertyUseType,"Main activity (office, hospital, warehouse...)..."


## Filtering and cleaning

Row-level rules, applied in order. Each filtering step prints how many rows it removes.

1. Keep non-residential buildings only.
2. Drop the outliers flagged by the city.
3. Drop invalid targets (missing or zero energy use).
4. Drop reports based on default data.
5. Drop the University of Washington campus.
6. Fix impossible values (0 floors, 0 buildings, a 99-floor church).
7. Harmonize the `Neighborhood` labels.

In [152]:
df = building_consumption.copy()
print(f"{'start':<35}{len(df):>6}")


def keep(df, mask, label):
    """Keep rows where mask is True and report how many were removed."""
    out = df[mask]
    print(f"{label:<35}{len(out):>6}  (-{len(df) - len(out)})")
    return out

start                                3376


### Step 1 – Non-residential only

`BuildingType` removes most of the housing, but a few rows labeled non-residential are still housing by their main use.

In [153]:
df = keep(
    df, ~df["BuildingType"].str.contains("Multifamily"), "non-residential building type"
)

multifamily_use = df["PrimaryPropertyType"].str.contains("Multifamily") | (
    df["LargestPropertyUseType"] == "Multifamily Housing"
)
display(
    df[multifamily_use]
    .groupby(["BuildingType", "PrimaryPropertyType", "LargestPropertyUseType"])
    .size()
    .rename("rows")
)
df = keep(df, ~multifamily_use, "no multifamily use left")

non-residential building type        1668  (-1708)


BuildingType    PrimaryPropertyType   LargestPropertyUseType
Campus          Low-Rise Multifamily  Multifamily Housing       1
NonResidential  Low-Rise Multifamily  Multifamily Housing       2
                Mixed Use Property    Multifamily Housing       9
Name: rows, dtype: int64

no multifamily use left              1656  (-12)


**Decision: drop them.** They are housing (low-rise multifamily, or mixed use where housing is the largest use), outside the scope.

### Step 2 – Outliers flagged by the city

In [154]:
display(df["Outlier"].value_counts())
display(
    df.loc[
        df["Outlier"] == "High outlier",
        [
            "PropertyName",
            "PrimaryPropertyType",
            "ListOfAllPropertyUseTypes",
            "SiteEUI(kBtu/sf)",
            "ComplianceStatus",
        ],
    ]
)
df = keep(df, df["Outlier"].isna(), "not flagged as outlier")

Outlier
Low outlier     14
High outlier     2
Name: count, dtype: int64

,PropertyName,PrimaryPropertyType,ListOfAllPropertyUseTypes,SiteEUI(kBtu/sf),ComplianceStatus
226,Second And Spring Building,Large Office,"Data Center, Office, Parking",297.399994,Non-Compliant
517,401 Elliott Ave West,Large Office,"Data Center, Office, Parking",311.700012,Non-Compliant


not flagged as outlier               1640  (-16)


**Decision: drop them.** The city flags reports whose site EUI is extremely low or high for their building type, as a sign of errors in the report.

The flag is a threshold per building type, so it does not always mean an error. The two high outliers are offices that also host a data center: their consumption is probably real (other buildings with a data center go well above 600 kBtu/sf without being flagged). They are non-compliant anyway, so step 4 would remove them too. The low outliers, with a site EUI mostly below 20 kBtu/sf, look like incomplete reports or nearly empty buildings.

This also shows that `PrimaryPropertyType` can hide an energy-intensive use. `ListOfAllPropertyUseTypes` is kept as a feature-engineering candidate for this reason.

### Step 3 – Valid target

We cannot learn from a building with no consumption. The `ComplianceStatus` of the removed rows shows that these reports were incomplete anyway.

In [155]:
valid_target = df[TARGET].notna() & (df[TARGET] > 0)
display(df.loc[~valid_target, "ComplianceStatus"].value_counts())
df = keep(df, valid_target, "target > 0")

ComplianceStatus
Missing Data                    14
Non-Compliant                    3
Error - Correct Default Data     1
Name: count, dtype: int64

target > 0                           1622  (-18)


### Step 4 – Default data

Some reports were filled in with default values when the owner did not provide the information.

In [156]:
display(pd.crosstab(df["ComplianceStatus"], df["DefaultData"]))
df = keep(df, df["ComplianceStatus"] == "Compliant", "compliant")

DefaultData,False,True
ComplianceStatus,,
Compliant,1537,0
Error - Correct Default Data,0,85


compliant                            1537  (-85)


**Decision: keep only `Compliant` reports.** The remaining non-compliant rows are exactly the ones with `DefaultData = True`, which is also why `DefaultData` is redundant with `ComplianceStatus`. The city considers reports with default values (mostly operating details such as hours or number of workers) as inaccurate and asks owners to correct them. Our main features are probably not affected, but we follow the city and keep only validated reports.

### Step 5 – Extreme target

In [157]:
df.nlargest(5, TARGET)[["OSEBuildingID", "PropertyName", "NumberofBuildings", TARGET]]

,OSEBuildingID,PropertyName,NumberofBuildings,SiteEnergyUse(kBtu)
3274,49967,University of Washington - Seattle Campus,111.0,873923712.0
35,43,Plant 2 Site,1.0,448385312.0
170,276,Harborview Medical Center,9.0,293090784.0
618,828,Swedish First Hill,7.0,291614432.0
558,753,WestinBuilding,1.0,274682208.0


In [158]:
df = keep(df, df["OSEBuildingID"] != 49967, "not the UW campus")

not the UW campus                    1536  (-1)


**Decision: drop the University of Washington campus.** Its consumption is almost twice the second highest, and it groups 111 buildings: it is a small town rather than a building, so it is outside what we want to predict.

### Step 6 – Impossible values

A building cannot have 0 floors or 0 buildings.

In [159]:
cols = [
    "OSEBuildingID",
    "PropertyName",
    "NumberofFloors",
    "NumberofBuildings",
    "PropertyGFABuilding(s)",
    TARGET,
]
display(df.loc[df["NumberofFloors"] == 0, cols])
print(f"Rows with 0 buildings: {(df['NumberofBuildings'] == 0).sum()}")

,OSEBuildingID,PropertyName,NumberofFloors,NumberofBuildings,PropertyGFABuilding(s),SiteEnergyUse(kBtu)
166,267,Grand Hyatt Seattle,0,1.0,934292,65047284.00
487,656,Arnold Pavilion,0,1.0,225982,20560622.00
488,657,2200 Westlake - SEDO,0,0.0,516407,39403320.00
564,758,Pacific Place,0,1.0,947987,46510960.00
1754,23311,HART First Hill LLC,0,1.0,274568,25311526.00
1993,24086,(ID#24086)Campus1:KC Metro Transit Atlantic Ce...,0,10.0,230971,21022288.00
3130,40028,Sandpoint #5,0,1.0,384772,15206762.00
3131,40031,Sandpoint #25,0,1.0,30287,2193115.00
3132,40034,Sandpoint #29,0,1.0,21931,3947209.25
3168,49705,Magnuson,0,8.0,502030,18470340.00


Rows with 0 buildings: 52


The buildings with 0 floors are large buildings with a real consumption (Grand Hyatt, Pacific Place...): 0 is a missing value, not a wrong row.

One building also stands out at the other end:

In [160]:
df.loc[df["NumberofFloors"] == df["NumberofFloors"].max(), cols]

,OSEBuildingID,PropertyName,NumberofFloors,NumberofBuildings,PropertyGFABuilding(s),SiteEnergyUse(kBtu)
1359,21611,Seattle Chinese Baptist Church,99,1.0,21948,326001.1875


**Decisions:**

- **0 floors → `NaN`**, imputed later in the scikit-learn pipeline. The imputed value (a median) is learned from the data, so it must be computed on the training set only, after the split.
- **0 buildings → 1.** This is a business rule, not a statistic: a reported property has at least one building. It learns nothing from the data, so it can be applied before the split.
- **The church listed with 99 floors → 1 floor.** It is a single-floor building (checked on Google Street View), which is consistent with its area and its energy use.

In [161]:
df["NumberofFloors"] = df["NumberofFloors"].replace(0, np.nan)
df["NumberofBuildings"] = df["NumberofBuildings"].replace(0, 1)
df.loc[df["OSEBuildingID"] == 21611, "NumberofFloors"] = (
    1  # Seattle Chinese Baptist Church
)

### Step 7 – Neighborhood labels

In [162]:
print(sorted(df["Neighborhood"].unique()))
df["Neighborhood"] = (
    df["Neighborhood"].str.upper().str.replace(" NEIGHBORHOODS", "", regex=False)
)
print(f"\nAfter cleaning: {df['Neighborhood'].nunique()} neighborhoods")

['BALLARD', 'Ballard', 'CENTRAL', 'Central', 'DELRIDGE', 'DELRIDGE NEIGHBORHOODS', 'DOWNTOWN', 'Delridge', 'EAST', 'GREATER DUWAMISH', 'LAKE UNION', 'MAGNOLIA / QUEEN ANNE', 'NORTH', 'NORTHEAST', 'NORTHWEST', 'North', 'Northwest', 'SOUTHEAST', 'SOUTHWEST']

After cleaning: 13 neighborhoods


**Decision:** upper case everywhere (`North` → `NORTH`) and drop the stray suffix (`DELRIDGE NEIGHBORHOODS` → `DELRIDGE`).

### Cleaned dataset

We keep only the target and the candidate features.

In [163]:
df = df[[TARGET, *KEY_FEATURES, *FE_CANDIDATES]].reset_index(drop=True)
print(f"Cleaned dataset: {df.shape[0]} rows x {df.shape[1]} columns")
df.isna().sum()[lambda s: s > 0]

Cleaned dataset: 1536 rows x 13 columns


NumberofFloors                   15
LargestPropertyUseType            4
SecondLargestPropertyUseType    704
dtype: int64